In [ ]:
# Las 3v clasicas

#Volumen = No caben en ram
#Velcidad = Sensores, clicks y logs
#Variedad = Diferentes formas en que llegan los datos y tipos.

In [1]:
print("Spark")

Spark


In [2]:
# SparkSession: punto de entrada de toda la clase.
# Transformaciones = select, filter, groupBy, join
# Acciones = show, count, collect, write

import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() 
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Tutoria Big Data Spark")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark listo")
print("version:", spark.version)
print("master:", spark.sparkContext.master)
print("ui:", spark.sparkContext.uiWebUrl)

c:\Users\JORGE\miniconda3\envs\py_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark listo
version: 4.2.0
master: local[*]
ui: http://ECHAVARRIA.mshome.net:4040


In [3]:
from pyarrow import schema

personas = spark.createDataFrame(
    [
        (1, "Ana", 32, "Espana"),
        (2, "Cristian", 28, "Colombia"),
        (3, "Marta", 50, "Espana"),
        (4, "Diego", 35, "Mexico"),
    ],
    schema=["id", "nombre", "edad", "pais"],
)

print("Columnas:", personas.columns)
print("Num filas:", personas.count())
print()
print("Esquema (nombre + tipo Spark)")
personas.printSchema()
print("Primeras filas (acción show):")
personas.show()

Columnas: ['id', 'nombre', 'edad', 'pais']
Num filas: 4

Esquema (nombre + tipo Spark)
root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- pais: string (nullable = true)

Primeras filas (acción show):
+---+--------+----+--------+
| id|  nombre|edad|    pais|
+---+--------+----+--------+
|  1|     Ana|  32|  Espana|
|  2|Cristian|  28|Colombia|
|  3|   Marta|  50|  Espana|
|  4|   Diego|  35|  Mexico|
+---+--------+----+--------+



In [4]:
ruta_ventas_csv = (RAIZ / "data" / "ventas.csv").resolve()

print("Vamos a leer este archivo:")
print(ruta_ventas_csv)
print("Existe:", ruta_ventas_csv.exists())

Vamos a leer este archivo:
C:\Programacion\DataScienceIA\Tutor_Dev_Senior_Code\Tutorias_Cohorte_6\Tutorias_Refuerzo_IA6\Tutoria_Big_data_spark\data\ventas.csv
Existe: True


In [13]:
ventas_inferido = (
    spark.read
    .option("header", "true")
    .option("inferSchema","true")
    .csv(ruta_ventas_csv.as_posix())
)

print("Tipo de objeto:", type(ventas_inferido).__name__)
print("Columnas que spark vio en la cabecera:", ventas_inferido.columns)
ventas_inferido.printSchema()

Tipo de objeto: DataFrame
Columnas que spark vio en la cabecera: ['venta_id', 'cliente_id', 'producto_id', 'fecha_venta', 'cantidad', 'precio_unitario', 'descuento', 'total', 'canal']
root
 |-- venta_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- producto_id: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- descuento: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)



In [7]:
n_filas = ventas_inferido.count()
print("Respuesat - numero de ventas del csv:", n_filas)

Respuesat - numero de ventas del csv: 200000


In [11]:
print("Muestra de 5 ventas")
ventas_inferido.show(5, truncate=False)

Muestra de 5 ventas
+--------+----------+-----------+-----------+--------+---------------+---------+-------+-----+
|venta_id|cliente_id|producto_id|fecha_venta|cantidad|precio_unitario|descuento|total  |canal|
+--------+----------+-----------+-----------+--------+---------------+---------+-------+-----+
|1       |2893      |1812       |2025-08-04 |2       |977.23         |0.2      |1563.57|web  |
|2       |1187      |1563       |2025-09-03 |3       |1083.68        |0.0      |3251.04|app  |
|3       |1053      |1184       |2025-11-02 |2       |134.73         |0.0      |269.46 |web  |
|4       |86        |1822       |2024-03-29 |2       |460.57         |0.1      |829.03 |web  |
|5       |9774      |1822       |2022-10-11 |1       |460.57         |0.0      |460.57 |web  |
+--------+----------+-----------+-----------+--------+---------------+---------+-------+-----+
only showing top 5 rows


## Segunda Lectura: Nosotros definimos el schema

In [15]:
esquema_ventas = StructType(
    [
        StructField("venta_id", IntegerType(), False),
        StructField("cliente_id", IntegerType(), False),
        StructField("producto_id", IntegerType(), False),
        StructField("fecha_venta", DateType(), False),
        StructField("cantidad", IntegerType(), False),
        StructField("precio_unitario", DoubleType(), False),
        StructField("descuento", DoubleType(), False),
        StructField("total", DoubleType(), False),
        StructField("canal", StringType(), False),
    ]
)


ventas = (
    spark.read
    .option("header", "True")
    .schema(esquema_ventas)
    .csv(ruta_ventas_csv.as_posix())
)

print("Receta (Schema)")
ventas.printSchema()


print("Respuesta -- Filas con esquema declarado:", ventas.count())
print("Este dataframe 'ventas' es el que vamos a guardar en el parquet")


Receta (Schema)
root
 |-- venta_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- producto_id: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- descuento: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)

Respuesta -- Filas con esquema declarado: 200000
Este dataframe 'ventas' es el que vamos a guardar en el parquet


### 1- Pasar el CSV a Parquet


### 2- Realizar busquedas o le pueden pedir a su motor IA que les genere ejercicios no resueltos que respondan a preguntas de negocio del CSV de ventas y así leer el parquet y generar las consultas.